In [39]:
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import  SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

MAKING THE MODEL

In [23]:
train_csv = pd.read_csv("kaggle folder/train.csv")
LoanDF = pd.DataFrame(train_csv)
LoanDF

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [24]:
#making a copy of the df and making ['Dependents'] into a number column instead of strings for the model
loan_clean = LoanDF.copy()
loan_clean.isna().sum()
loan_clean['Dependents'] = loan_clean['Dependents'].replace('3+', '3').astype(float)
loan_clean['Self_Employed'].unique()

array(['No', 'Yes', nan], dtype=object)

In [25]:
#creating the list of columns for encoding
number_columns = ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Credit_History']
text_columns = ['Education', 'Self_Employed']

In [26]:
#ColumnTransformer to ready our columns for the model
preprocessing = ColumnTransformer([
    ('num', SimpleImputer(strategy= 'median'), number_columns),
    ('cat', Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder()),
    ]), text_columns)
])

In [27]:
#creating the main pipeline to scale and predict the user input
pipeline = Pipeline([
    ('preprocess', preprocessing),
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', class_weight='balanced'))
])
#added class_weight = balanced since it still had some mistakes on FP which would cost the bank a lot of money
#but that actually changed nothing about the scores, but it's harmless so i'll keep it there

#making another one with svm rbf kernel for comparison
rbf_pipeline = Pipeline([
    ('preprocess', preprocessing),
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])
#after comparing both and getting almost the same score, i will stick with linear since it's the simpler one and more readable

In [28]:
#Defining my features and label
X = loan_clean[number_columns + text_columns]
y = loan_clean['Loan_Status']

In [29]:
#fitting both 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipeline.fit(X_train, y_train)
print(pipeline.score(X_test, y_test))

rbf_pipeline.fit(X_train, y_train)
print(rbf_pipeline.score(X_test, y_test))

print(y.value_counts(normalize=True))

#defining y_pred
y_pred = pipeline.predict(X_test)
y_pred

0.7837837837837838
0.7783783783783784
Loan_Status
Y    0.687296
N    0.312704
Name: proportion, dtype: float64


array(['Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'N', 'N', 'Y', 'Y',
       'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'N', 'Y', 'N', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'N',
       'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'Y',
       'N', 'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y',
       'Y', 'N', 'Y', 'N', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'Y', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y',
       'N', 'Y', 'Y', 'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y',
       'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N', 'Y', 'Y', 'Y',
       'Y', 'Y', 'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'N',
       'Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y

In [30]:
#getting the scores
print(precision_score(y_test, y_pred, pos_label='Y'))
print(accuracy_score(y_test, y_pred))
print(recall_score(y_test, y_pred, pos_label='Y'))
print(f1_score(y_test, y_pred, pos_label='Y'))
print(confusion_matrix(y_test, y_pred))

0.7564102564102564
0.7837837837837838
0.9833333333333333
0.855072463768116
[[ 27  38]
 [  2 118]]


In [31]:
#testing to see if something was changed on the scores or maybe the pipeline didn't even change
print(pipeline.named_steps['svm'].class_weight)
print(confusion_matrix(y_test, y_pred))
#balanced, no change. might be an underlying issue with one of the features, i'll leave it for now.

balanced
[[ 27  38]
 [  2 118]]


In [38]:
#saving samples for the model info dump
samples_x = X_test.astype(object).where(X_test.notna(), None).head().to_dict(orient="records")
samples_y = y_test.head()

samples = []
for row, label in zip(samples_x, samples_y):
    row['Loan_Status'] = label
    samples.append(row)
samples

[{'Dependents': 0.0,
  'ApplicantIncome': 9083,
  'CoapplicantIncome': 0.0,
  'LoanAmount': 228.0,
  'Credit_History': 1.0,
  'Education': 'Graduate',
  'Self_Employed': 'No',
  'Loan_Status': 'Y'},
 {'Dependents': 0.0,
  'ApplicantIncome': 4310,
  'CoapplicantIncome': 0.0,
  'LoanAmount': 130.0,
  'Credit_History': None,
  'Education': 'Graduate',
  'Self_Employed': 'No',
  'Loan_Status': 'Y'},
 {'Dependents': 2.0,
  'ApplicantIncome': 4167,
  'CoapplicantIncome': 1447.0,
  'LoanAmount': 158.0,
  'Credit_History': 1.0,
  'Education': 'Graduate',
  'Self_Employed': 'No',
  'Loan_Status': 'Y'},
 {'Dependents': 0.0,
  'ApplicantIncome': 2900,
  'CoapplicantIncome': 0.0,
  'LoanAmount': 71.0,
  'Credit_History': 1.0,
  'Education': 'Graduate',
  'Self_Employed': 'No',
  'Loan_Status': 'Y'},
 {'Dependents': 0.0,
  'ApplicantIncome': 2718,
  'CoapplicantIncome': 0.0,
  'LoanAmount': 70.0,
  'Credit_History': 1.0,
  'Education': 'Graduate',
  'Self_Employed': 'No',
  'Loan_Status': 'Y'}]

In [42]:
#saving the pipeline
joblib.dump(pipeline, "loan_model.joblib")

#saving the model info
info_dict = {"accuracy":  float(accuracy_score(y_test, y_pred)),
             "precision": float(precision_score(y_test, y_pred, pos_label='Y')),
             "recall": float(recall_score(y_test, y_pred, pos_label='Y')),
             "f1 score": float(f1_score(y_test, y_pred, pos_label='Y')),
             "confusion matrix": confusion_matrix(y_test, y_pred).tolist(),
             "number features": number_columns,
             "text features": text_columns,
             "classes": pipeline.classes_.tolist(),
             "model samples": samples,
             "file name": "loan_model.joblib"
            }
json.dumps(info_dict)

with open("model_info.json", "w") as f:
    json.dump(info_dict, f)